# Chapter 4 — Multiple Regression: Inference
## University of Neuchâtel — Python for Econometrics

Topics:
1. Sampling distribution of OLS estimators
2. Standard errors, t-statistics and p-values
3. Confidence intervals
4. F-test for overall significance
5. F-test for joint hypotheses
6. Monte Carlo: empirical sampling distribution

**Format:** One cell per step. Manual first, statsmodels second, `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Data and Run OLS

In [ ]:
df = woo.dataWoo('wage1')
print('Shape:', df.shape)
print(df[['wage','educ','exper','tenure']].head(8))

In [ ]:
n = len(df)
k = 3  # regressors (excl. constant)
X = np.column_stack([np.ones(n), df['educ'].values, df['exper'].values, df['tenure'].values])
y = df['wage'].values
labels = ['const','educ','exper','tenure']
beta_hat = np.linalg.solve(X.T @ X, X.T @ y)
print('OLS coefficients:', beta_hat.round(4))

## 2 — Standard Errors

Under Gauss-Markov assumptions:

$$\widehat{Var}(\hat{\boldsymbol{\beta}}) = \hat{\sigma}^2 (X'X)^{-1}, \quad \hat{\sigma}^2 = \frac{SSR}{n-k-1}$$

### Step 2.1 — sigma² and variance matrix manually

In [ ]:
e       = y - X @ beta_hat
SSR     = np.sum(e**2)
sig2    = SSR / (n - k - 1)
V_hat   = sig2 * np.linalg.inv(X.T @ X)
SE_man  = np.sqrt(np.diag(V_hat))
print('Manual standard errors:')
for lbl, se in zip(labels, SE_man):
    print(f'  {lbl:8s}: {se:.4f}')

### Step 2.2 — Statsmodels OLS

In [ ]:
res = smf.ols('wage ~ educ + exper + tenure', data=df).fit()
print(res.summary())

### Step 2.3 — Compare SEs

In [ ]:
print('SE comparison:')
for lbl, se_man in zip(labels, SE_man):
    se_sm = res.bse[lbl]
    print(f'  {lbl:8s}: manual={se_man:.4f}, sm={se_sm:.4f}, match={np.isclose(se_man,se_sm)}')

## 3 — t-Statistics

$$t_j = \frac{\hat{\beta}_j - 0}{SE(\hat{\beta}_j)} \sim t_{n-k-1} \text{ under } H_0: \beta_j=0$$

### Step 3.1 — t-statistics manually

In [ ]:
t_man = beta_hat / SE_man
print('Manual t-statistics:')
for lbl, t in zip(labels, t_man):
    print(f'  {lbl:8s}: {t:.4f}')

### Step 3.2 — Compare t-statistics

In [ ]:
print('t-stat comparison:')
for lbl, t_m in zip(labels, t_man):
    t_sm = res.tvalues[lbl]
    print(f'  {lbl:8s}: manual={t_m:.4f}, sm={t_sm:.4f}, match={np.isclose(t_m,t_sm)}')

## 4 — p-values

Two-sided p-value:

$$p = 2 \cdot P(T_{n-k-1} > |t|) = 2 \cdot \text{sf}(|t|, n-k-1)$$

### Step 4.1 — p-values manually

In [ ]:
df_resid = n - k - 1
pval_man = 2 * stats.t.sf(np.abs(t_man), df=df_resid)
print('Manual p-values:')
for lbl, p in zip(labels, pval_man):
    sig = '***' if p<0.01 else ('**' if p<0.05 else ('*' if p<0.1 else ''))
    print(f'  {lbl:8s}: {p:.4f} {sig}')

### Step 4.2 — Compare p-values

In [ ]:
print('p-value comparison:')
for lbl, p_m in zip(labels, pval_man):
    p_sm = res.pvalues[lbl]
    print(f'  {lbl:8s}: manual={p_m:.4f}, sm={p_sm:.4f}, match={np.isclose(p_m,p_sm)}')

## 5 — Confidence Intervals

$$95\%\ CI: \left[\hat{\beta}_j - t_{0.975,n-k-1} \cdot SE_j,\ \hat{\beta}_j + t_{0.975,n-k-1} \cdot SE_j\right]$$

### Step 5.1 — 95% CI manually

In [ ]:
t_crit = stats.t.ppf(0.975, df=df_resid)
print(f'Critical value t_{{0.975, {df_resid}}} = {t_crit:.4f}')
CI_lower = beta_hat - t_crit * SE_man
CI_upper = beta_hat + t_crit * SE_man
print('95% Confidence Intervals (manual):')
for lbl, lo, hi in zip(labels, CI_lower, CI_upper):
    print(f'  {lbl:8s}: [{lo:.4f}, {hi:.4f}]')

### Step 5.2 — Compare CIs with statsmodels

In [ ]:
ci_sm = res.conf_int()
print('CI comparison:')
for i, lbl in enumerate(labels):
    lo_man, hi_man = CI_lower[i], CI_upper[i]
    lo_sm,  hi_sm  = ci_sm.loc[lbl]
    match_lo = np.isclose(lo_man, lo_sm)
    match_hi = np.isclose(hi_man, hi_sm)
    print(f'  {lbl:8s}: manual=[{lo_man:.4f},{hi_man:.4f}], match=[{match_lo},{match_hi}]')

### Step 5.3 — Visualise coefficient estimates with CI

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
y_pos = range(len(labels))
ax.errorbar(beta_hat, y_pos,
            xerr=[beta_hat - CI_lower, CI_upper - beta_hat],
            fmt='o', capsize=5, linewidth=2)
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('Coefficient estimate')
ax.set_title('OLS Coefficients with 95% CI')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch04_01_coeff_ci.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6 — F-test: Overall Significance

$H_0$: $\beta_1 = \beta_2 = \beta_3 = 0$ (all regressors jointly zero)

$$F = \frac{SSE/k}{SSR/(n-k-1)} \sim F_{k, n-k-1} \text{ under } H_0$$

### Step 6.1 — F-statistic manually

In [ ]:
y_mean = y.mean()
SST = np.sum((y - y_mean)**2)
SSE = SST - SSR
F_manual = (SSE / k) / (SSR / (n - k - 1))
pval_F = stats.f.sf(F_manual, dfn=k, dfd=n-k-1)
print(f'F-statistic (manual): {F_manual:.4f}')
print(f'p-value:              {pval_F:.6f}')

### Step 6.2 — Compare F-stat with statsmodels

In [ ]:
print(f'Statsmodels F-stat: {res.fvalue:.4f}')
print(f'Match: {np.isclose(F_manual, res.fvalue)}')
print('Reject H0 → at least one regressor significantly explains wage')

## 7 — F-test: Joint Hypothesis

$H_0$: $\beta_{exper} = 0$ AND $\beta_{tenure} = 0$ (both jointly zero)

**Restricted model:** $wage = \beta_0 + \beta_1 educ + u$

$$F = \frac{(SSR_R - SSR_{UR})/q}{SSR_{UR}/(n-k-1)}$$

where $q=2$ restrictions.

### Step 7.1 — Restricted model

In [ ]:
res_r = smf.ols('wage ~ educ', data=df).fit()
SSR_R = res_r.ssr
SSR_UR = res.ssr
q = 2
print(f'SSR restricted:   {SSR_R:.4f}')
print(f'SSR unrestricted: {SSR_UR:.4f}')

### Step 7.2 — Manual F-stat for joint test

In [ ]:
F_joint = ((SSR_R - SSR_UR) / q) / (SSR_UR / (n - k - 1))
pval_joint = stats.f.sf(F_joint, dfn=q, dfd=n-k-1)
print(f'F-stat (exper=0, tenure=0): {F_joint:.4f}')
print(f'p-value: {pval_joint:.6f}')
if pval_joint < 0.05:
    print('Reject H0: exper and tenure are jointly significant')

### Step 7.3 — Joint test with statsmodels f_test

In [ ]:
f_test = res.f_test('exper=0, tenure=0')
print(f'statsmodels f_test:')
print(f'  F-stat: {f_test.fvalue[0][0]:.4f}')
print(f'  p-val:  {f_test.pvalue:.6f}')
print(f'  Match: {np.isclose(F_joint, f_test.fvalue[0][0])}')

## 8 — Monte Carlo: Sampling Distribution of b1

In [ ]:
np.random.seed(42)
n_mc = 200
n_sim = 5000
true_b1 = 0.5
x_mc = np.random.normal(12, 2, n_mc)
b1_estimates = np.zeros(n_sim)
for s in range(n_sim):
    u = np.random.normal(0, 2, n_mc)
    y_mc = 1 + true_b1 * x_mc + u
    b1_estimates[s] = np.sum((x_mc - x_mc.mean())*(y_mc - y_mc.mean())) / np.sum((x_mc - x_mc.mean())**2)
print(f'True b1:        {true_b1}')
print(f'Mean b1_hat:    {b1_estimates.mean():.4f}  (should be close to {true_b1})')
print(f'Std b1_hat:     {b1_estimates.std():.4f}  (empirical SE)')

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(b1_estimates, bins=60, density=True, alpha=0.7, edgecolor='black')
x_range = np.linspace(b1_estimates.min(), b1_estimates.max(), 200)
ax.plot(x_range, stats.norm.pdf(x_range, true_b1, b1_estimates.std()), 'r-', linewidth=2, label='Normal approximation')
ax.axvline(true_b1, color='green', linestyle='--', linewidth=2, label=f'True b1={true_b1}')
ax.set_xlabel('b1 estimate')
ax.set_ylabel('Density')
ax.set_title('Monte Carlo: Sampling Distribution of b1 (n=5000 simulations)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch04_02_sampling_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print('CLT: as n increases, the sampling distribution approaches Normal')

## 9 — Summary Table

In [ ]:
sumtab = pd.DataFrame({
    'Variable': labels,
    'beta':     beta_hat,
    'SE':       SE_man,
    't_stat':   t_man,
    'p_value':  pval_man,
    'CI_lower': CI_lower,
    'CI_upper': CI_upper
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch04_inference.csv'), index=False)
print('Saved table_ch04_inference.csv')